# 快速入门

[ReAct Agent](https://docs.langchain.com/oss/python/langchain/agents#tool-use-in-the-react-loop) 是一种将 **推理**（Reasoning）与 **行动**（Acting）结合起来的智能体。它是智能体的核心科技，也是智能体框架中最能表现其自主性的组件。

它的工作流程遵循三步循环：

1. 思考
2. 行动
3. 观察

这个循环会持续进行，直到 LLM 判断任务已经完成或无法继续。

这意味着智能体可以通过工具调用，自动补足当前未知的上下文信息。然后基于新获取的信息，做出下一步决策。比如你要智能体查询一个数据表中的记录，它可能尚不知道数据库中有哪些表，表中有哪些字段。但是通过几轮主动查询与观察，即使你询问的信息比较模糊，它大概率也能从表名和字段中推测出你需要的记录是哪一条。这就是 ReAct Agent 的威力。

本节将介绍：

- 如何创建简单的 ReAct Agent
- 如何创建带工具的 ReAct Agent
- 如何创建带工具权限的 ReAct Agent
- 结构化输出
- 流式输出


## 一、环境配置

### 1）安装依赖

你可以下载 [本仓库](https://github.com/luochang212/dive-into-langgraph) 到本地，然后运行以下命令，安装完整的 Python 依赖：

```bash
cd dive-into-langgraph
pip install -r requirements.txt
```

### 2）配置 API Key

使用 .env.example 创建 .env 文件:

```bash
cp .env.example .env
```

> PS: This local version reads the repository-root `.env`. Set `DIVE_PROVIDER` or `LANGCHAIN_PROVIDER` to choose `dashscope`, `deepseek`, `openai`, `ark`, or `ollama`, then fill in the matching API key variables.


In [ ]:
import os

from IPython.display import HTML, display, update_display
from langchain.agents import create_agent

# Load shared project config from the repository-root .env
from pathlib import Path
import sys


def find_dive_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "dive_config.py").exists():
            return candidate
        nested = candidate / "LangChain" / "projects" / "dive-into-langgraph"
        if (nested / "dive_config.py").exists():
            return nested
    raise RuntimeError(
        "Start this notebook from the repository root or the "
        "dive-into-langgraph project directory."
    )


PROJECT_ROOT = find_dive_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dive_config import create_chat_model, create_embeddings, get_model_config, load_project_env

load_project_env(PROJECT_ROOT)
model_config = get_model_config()


### 3）加载 LLM

下面是加载大模型的方法：

In [ ]:
# Load LLM from shared project config
llm = create_chat_model()


## 二、简单的 Agent

首先，创建一个最简单的 ReAct Agent。

In [ ]:
# 创建一个简单的Agent
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
)

# 运行Agent
response = agent.invoke({'messages': '你好'})

response['messages'][-1].content


In [ ]:
# 可视化 Agent
agent


## 三、带工具调用的 Agent

接下来，我们创建一个带工具调用的 ReAct Agent，它会根据需求自主决定是否调用工具。

In [ ]:
# 一个工具函数
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's sunny in {city}!"

# 创建带工具调用的Agent
tool_agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# 运行Agent
response = tool_agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

response['messages'][-1].content


In [ ]:
# 可视化 Agent
tool_agent


## 四、使用 `ToolRuntime` 控制工具权限

下面创建一个带 runtime 的工具，runtime 类型为 ToolRuntime。当我们调用 Agent 时，会将这个 runtime 传递给工具。工具再根据 runtime 中的信息，判断当前调用是否具备执行权限。

In [ ]:
from typing import Literal, Any
from pydantic import BaseModel
from langchain.tools import tool, ToolRuntime

class Context(BaseModel):
    authority: Literal["admin", "user"]

# 创建带权限控制的tool，依赖ToolRuntime的内容进行判断
@tool
def math_add(runtime: ToolRuntime[Context, Any], a: int, b: int) -> int:
    """Add two numbers together."""
    authority = runtime.context.authority
    # 只有admin用户可以访问加法工具
    if authority != "admin":
        raise PermissionError("User does not have permission to add numbers")
    return a + b

# 创建带工具调用的Agent
tool_agent = create_agent(
    model=llm,
    tools=[get_weather, math_add],
    system_prompt="You are a helpful assistant",
)

# 在运行Agent时注入context
response = tool_agent.invoke(
    {"messages": [{"role": "user", "content": "请计算 8234783 + 94123832 = ?"}]},
    config={"configurable": {"thread_id": "1"}},
    context=Context(authority="admin"),
)


In [ ]:
for message in response['messages']:
    message.pretty_print()


In [ ]:
# 验证计算结果是否正确
8234783 + 94123832


## 五、结构化输出

若想获得 [结构化输出](https://docs.langchain.com/oss/python/langchain/structured-output#response-format)（Structured Output），可以在 create_agent 函数的 response_format 参数进行设定。在下面的例子中，我们用 BaseModel 定义输出格式，然后在 response_format 中指定该格式。

In [ ]:
from pydantic import BaseModel, Field

class CalcInfo(BaseModel):
    """Calculation information."""
    output: int = Field(description="The calculation result")


In [ ]:
# 创建带结构化输出的Agent
structured_agent = create_agent(
    model=llm,
    tools=[get_weather, math_add],
    system_prompt="You are a helpful assistant",
    response_format=CalcInfo,
)

response = structured_agent.invoke(
    {"messages": [{"role": "user", "content": "请计算 8234783 + 94123832 = ?"}]},
    config={"configurable": {"thread_id": "1"}},
    context=Context(authority="admin"),
)


In [ ]:
for message in response['messages']:
    message.pretty_print()


In [ ]:
response['structured_response']


## 六、流式输出

下面是简单介绍，更多信息请参阅 [streaming](https://docs.langchain.com/oss/python/langchain/streaming/overview).

### 1）`updates` 模式

`stream_mode="updates"`：每个节点完成后流式更新

In [ ]:
%autoawait on

agent = create_agent(
    model=llm,
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")


### 2）`messages` 模式

`stream_mode="messages"`：每个 token 生成后流式更新

In [ ]:
async def agent_invoke():
    content, last_node = "", ""
    display_handle = display("", display_id=True)

    async for token, metadata in agent.astream(  
        {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
        stream_mode="messages",
    ):
        node = metadata['langgraph_node']
        if last_node and node != last_node:
            content += f"\n\n🔹 <b>node:</b> {node}\n📝 <b>content: </b>"
        if node == 'model' and token.content:
            content += token.content
        elif node == 'tools' and token.content:
            content += token.content

        last_node = node
        update_display(HTML(f"<pre>{content}</pre>"), display_id=display_handle.display_id)

print("打印工具节点和模型节点的输出结果：")
await agent_invoke()


### 3）`values` 模式

`stream_mode="values"`：流式输出 State 的快照，可查看工具调用信息

In [ ]:
async def agent_invoke():
    async for payload in agent.astream(  
        {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
        stream_mode="messages",
    ):
        state = payload[0]
        if hasattr(state, "tool_calls") and state.tool_calls:
            if state.tool_calls[0].get('name'):
                print("name:", state.tool_calls[0].get('name'))
            if state.tool_calls[0].get('args'):
                print("args:", state.tool_calls[0].get('args'))

print("打印工具调用信息：\n")
await agent_invoke()
